# Notebook 29 — Persistence Layer

All in-memory stores in Multigen are ephemeral.  The `persistence` module
provides **SQLite-backed** (zero extra dependencies) drop-in replacements that
survive process restarts.

| Class | Replaces | Persists |
|---|---|---|
| `SQLiteSessionStore` | `InMemorySessionStore` | sessions |
| `SQLiteMemoryStore` | `ShortTermMemory` | key-value memory |
| `PersistentEpisodicMemory` | `EpisodicMemory` | episode log |
| `SQLiteCheckpointStore` | — | workflow step checkpoints |
| `CheckpointedRuntime` | `Runtime` | crash-resumable chains |
| `DurableQueue` | in-memory `BatchExecutor` queue | task queue |

## 1. Setup

In [ ]:
import asyncio, tempfile, pathlib
from multigen.persistence import (
    SQLiteSessionStore, SQLiteMemoryStore, PersistentEpisodicMemory,
    SQLiteCheckpointStore, CheckpointedRuntime, DurableQueue,
    Checkpoint, open_persistent_stores,
)
from multigen.session import SessionManager, SessionState

TMPDIR = tempfile.mkdtemp()
print('Working directory:', TMPDIR)

## 2. Persistent Sessions

`SQLiteSessionStore` is a drop-in replacement for `InMemorySessionStore`.
All sessions survive a Python process restart.

In [ ]:
async def demo_sessions():
    store   = SQLiteSessionStore(f'{TMPDIR}/sessions.db')
    manager = SessionManager(store=store, default_ttl=3600)

    sess = await manager.create_session('research-agent', metadata={'user': 'alice'})
    print(f'Created session: {sess.session_id}  state={sess.state}')

    # Touch (refreshes last_active)
    await manager.touch_session(sess.session_id)

    # Reload from disk — simulates a restart
    store2 = SQLiteSessionStore(f'{TMPDIR}/sessions.db')
    fetched = await store2.get(sess.session_id)
    print(f'Fetched from new store: {fetched.session_id}  agent={fetched.agent_id}')

    # List active sessions
    active = await manager.list_active()
    print(f'Active sessions: {len(active)}')

    store.close(); store2.close()

await demo_sessions()

## 3. Persistent Memory

`SQLiteMemoryStore` extends `ShortTermMemory` with write-through SQLite
backing.  On instantiation it warms the in-process cache from the DB.

In [ ]:
def demo_memory():
    mem = SQLiteMemoryStore(f'{TMPDIR}/memory.db', default_ttl=600)
    mem.store('capital_of_france', 'Paris', tags=['geography'])
    mem.store('pi', 3.14159)
    mem.store('config', {'debug': True, 'retries': 3})

    # Simulate restart — open new instance on same DB
    mem2 = SQLiteMemoryStore(f'{TMPDIR}/memory.db')
    print('capital_of_france:', mem2.retrieve('capital_of_france'))
    print('pi:', mem2.retrieve('pi'))
    print('config:', mem2.retrieve('config'))
    print('stats:', mem2.stats())
    mem.close(); mem2.close()

demo_memory()

## 4. Persistent Episodic Memory

`PersistentEpisodicMemory` writes every recorded episode to SQLite.
All episodes are reloaded on startup.

In [ ]:
def demo_episodes():
    ep = PersistentEpisodicMemory(f'{TMPDIR}/episodes.db', max_episodes=1000)

    for i in range(5):
        ep.record(
            agent=f'AgentStep{i}',
            inputs={'step': i, 'query': f'question {i}'},
            outputs={'answer': f'answer {i}', 'confidence': 0.9 - i * 0.1},
            tags=['demo', f'step-{i}'],
        )
    print(f'Recorded {len(ep)} episodes')

    # Simulate restart
    ep2 = PersistentEpisodicMemory(f'{TMPDIR}/episodes.db')
    print(f'After restart: {len(ep2)} episodes loaded')
    recent = ep2.recent(3)
    for e in recent:
        print(f'  {e.agent}: inputs={e.inputs}')

    results = ep2.search('question 2')
    print(f'Search "question 2": {len(results)} match(es)')
    ep.close(); ep2.close()

demo_episodes()

## 5. Checkpoint Store

`SQLiteCheckpointStore` records each completed step.  You can inspect what ran
and resume from any point.

In [ ]:
async def demo_checkpoints():
    store = SQLiteCheckpointStore(f'{TMPDIR}/checkpoints.db')

    run_id = 'report-pipeline-001'
    steps  = [
        Checkpoint(run_id, 0, 'DataLoader',   {'raw': True},          {'rows': 500}),
        Checkpoint(run_id, 1, 'Preprocessor', {'rows': 500},          {'clean': True}),
        Checkpoint(run_id, 2, 'Analyser',     {'clean': True},        {'insights': ['trend A', 'trend B']}),
    ]
    for ckpt in steps:
        await store.save(ckpt)

    last = await store.last_completed(run_id)
    print(f'Last completed step: {last.step_index} ({last.agent_name})')

    all_steps = await store.get_run(run_id)
    for s in all_steps:
        print(f'  step {s.step_index}: {s.agent_name}  result={s.result_snapshot}')

    runs = await store.list_runs()
    print(f'Known runs: {runs}')
    store.close()

await demo_checkpoints()

## 6. CheckpointedRuntime — crash & resume

`CheckpointedRuntime` wraps any list of agents and checkpoints each step.
If the process crashes mid-chain you can call `resume()` to skip already-
completed steps.

In [ ]:
class StepAgent:
    def __init__(self, name, key, value):
        self.name  = name
        self._key  = key
        self._val  = value
    async def __call__(self, ctx):
        print(f'  Running {self.name}')
        return {self._key: self._val}

async def demo_runtime():
    rt = CheckpointedRuntime(db_path=f'{TMPDIR}/rt_demo.db')

    agents = [
        StepAgent('Loader',  'loaded',    True),
        StepAgent('Parser',  'parsed',    {'items': 3}),
        StepAgent('Writer',  'written',   True),
    ]

    print('── First run ──')
    result = await rt.run_chain(agents, ctx={'job': 'ingest'}, run_id='ingest-01')
    print('Result:', result)

    status = await rt.run_status('ingest-01')
    print('Status:', status)

await demo_runtime()

In [ ]:
async def demo_resume():
    """Simulate a crash after step 1 by manually creating checkpoints."""
    ckpt_store = SQLiteCheckpointStore(f'{TMPDIR}/rt_resume.db')
    await ckpt_store.save(Checkpoint('resume-run', 0, 'StepA', {'x': 1}, {'a': 'done'}))
    # Step 1 never ran — simulate crash

    rt = CheckpointedRuntime(checkpoint_store=ckpt_store)
    agents = [
        StepAgent('StepA', 'a', 'done'),   # already done
        StepAgent('StepB', 'b', 'done'),   # will run
        StepAgent('StepC', 'c', 'done'),   # will run
    ]

    print('── Resuming from checkpoint ──')
    result = await rt.resume('resume-run', agents)
    print('Resumed result:', {k: v for k, v in result.items() if not k.startswith('_')})
    ckpt_store.close()

await demo_resume()

## 7. DurableQueue

`DurableQueue` is a crash-safe FIFO queue.  Items remain in-flight until
explicitly `ack()`-ed.  Stuck in-flight items can be recovered with
`recover_pending()`.

In [ ]:
async def demo_queue():
    q = DurableQueue(f'{TMPDIR}/tasks.db')

    # Enqueue work items (higher priority = dequeued first)
    for i, prio in [(1, 0), (2, 5), (3, 0)]:
        await q.enqueue({'task': 'summarise', 'doc_id': i}, priority=prio)

    print('Queue size (pending):', await q.size())
    print('Stats:', await q.stats())

    # Priority-ordered dequeue
    item = await q.dequeue()    # doc_id=2 (priority=5)
    print('Dequeued (highest priority):', item)
    await q.ack(item['_queue_id'])

    item2 = await q.dequeue()
    print('Dequeued next:', item2)
    # Simulate crash — nack to retry
    await q.nack(item2['_queue_id'])
    print('Stats after nack:', await q.stats())

    # Recover stuck in-flight (items dequeued >0 seconds ago)
    recovered = await q.recover_pending(older_than=0)
    print(f'Recovered {recovered} stuck item(s)')
    print('Final stats:', await q.stats())
    q.close()

await demo_queue()

## 8. open_persistent_stores — all stores in one call

Convenience factory that opens all five stores under a common directory.

In [ ]:
stores = open_persistent_stores(f'{TMPDIR}/data', prefix='demo')
print('Opened stores:', list(stores.keys()))

# Use them normally
stores['memory'].store('hello', 'world')
print('Memory recall:', stores['memory'].retrieve('hello'))

for s in stores.values():
    s.close()
print('All stores closed cleanly')

## 9. Integration — full crash-resilient pipeline

Combine a durable queue as input feeder with a checkpointed runtime so that
both task scheduling and step execution are crash-resilient end-to-end.

In [ ]:
async def full_integration():
    stores = open_persistent_stores(f'{TMPDIR}/prod', prefix='pipeline')
    queue  = stores['queue']
    mem    = stores['memory']
    ep     = stores['episodes']

    # Feed work into the durable queue
    for doc in ['report-a', 'report-b', 'report-c']:
        await queue.enqueue({'doc': doc}, priority=1)

    rt = CheckpointedRuntime(checkpoint_store=stores['checkpoints'])

    class DocProcessor:
        def __init__(self): self.name = 'DocProcessor'
        async def __call__(self, ctx):
            doc = ctx.get('doc', 'unknown')
            result = {'processed': doc, 'word_count': len(doc) * 10}
            mem.store(f'result:{doc}', result)
            ep.record('DocProcessor', inputs=ctx, outputs=result)
            return result

    processed = 0
    while True:
        item = await queue.dequeue()
        if item is None:
            break
        run_id = f"run-{item['doc']}"
        ctx    = {'doc': item['doc']}
        result = await rt.run_chain([DocProcessor()], ctx=ctx, run_id=run_id)
        await queue.ack(item['_queue_id'])
        processed += 1
        print(f'Processed: {result["processed"]}  words≈{result["word_count"]}')

    print(f'\nTotal processed: {processed}')
    print(f'Episodes recorded: {len(ep)}')
    print(f'Memory keys: {mem.keys()}')

    for s in stores.values():
        s.close()

await full_integration()

## Summary

| Component | Persistence | Crash-safe | Zero deps |
|---|---|---|---|
| `SQLiteSessionStore` | SQLite WAL | yes | yes |
| `SQLiteMemoryStore` | SQLite WAL | yes | yes |
| `PersistentEpisodicMemory` | SQLite WAL | yes | yes |
| `SQLiteCheckpointStore` | SQLite WAL | yes | yes |
| `CheckpointedRuntime` | via `SQLiteCheckpointStore` | yes | yes |
| `DurableQueue` | SQLite WAL | ack/nack + recover | yes |

All stores:
- Use **WAL journal mode** for write performance
- Are **async-safe** via `asyncio.Lock`
- Support `.close()` for clean shutdown
- Accept any file path — use `:memory:` for tests